# 02 - After-Closure Analysis (ACA)

# DFIT Pressure Diagnostics
Developed as part of DFIT pressure-diagnostics research in the Harold Vance Department of Petroleum Engineering at Texas A&M University.

Primary reference: Barree, Miskimins & Gilbert (2015), SPE-169539-PA.

Once the fracture closes, the reservoir controls the falloff. The log-log derivative of pressure change vs shut-in time reveals the flow regime through its slope: about -1/2 for pseudo-linear flow and about -1 for pseudo-radial flow.

Barree et al. (2015) stress that true pseudo-radial flow is the exception in tight rock - it can take hundreds of days to develop - so forcing a radial (Horner) interpretation usually overstates permeability by orders of magnitude. This notebook shows how to read the regime honestly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import dfit
from dfit.afterclosure import aca_derivative, detect_flow_regimes

## 1. A test that does reach linear flow

In [ ]:
d = dfit.generate_dfit(regime='normal', after_closure_regime='linear',
                       closure_G=6.0, n_points=1500, G_max=40, seed=4)
res = dfit.analyze_dfit(d.time_min, d.pressure_psi, d.rate_bpm)
t_closure = res.closure_time_min or 50.0
dt, dp, deriv = aca_derivative(d.time_min, d.pressure_psi, t_closure)
flow = detect_flow_regimes(dt, dp, deriv)
print('linear window:', flow.linear_flow_window)
print('radial window:', flow.radial_flow_window)
print('radial supported:', flow.radial_supported)
print(flow.note)

In [ ]:
mask = np.isfinite(dt) & np.isfinite(deriv) & (deriv>0)
plt.figure(figsize=(7,5))
plt.loglog(dt[mask], dp[mask], 'b.', ms=3, label='dp = p - p_far')
plt.loglog(dt[mask], deriv[mask], 'r.', ms=3, label='log-log derivative')
# reference -1/2 slope guide
xg = np.array([dt[mask].min(), dt[mask].max()])
plt.loglog(xg, deriv[mask][len(deriv[mask])//2]*(xg/np.median(dt[mask]))**-0.5,
           'k--', lw=1, label='-1/2 slope guide')
plt.xlabel('shut-in time (min)'); plt.ylabel('pressure / derivative (psi)')
plt.title('After-closure log-log diagnostic')
plt.legend(fontsize=8); plt.grid(alpha=0.3, which='both'); plt.tight_layout(); plt.show()

## 2. The Barree warning: do not force radial flow

When the radial-looking section is short or appears too early, the detector reports `radial_supported = False`. Honouring that flag is the single most important guard against the over-estimated permeability that the paper documents.

In [ ]:
# a tight case where reservoir flow never really develops
d2 = dfit.generate_dfit(regime='normal', after_closure_regime='linear',
                        closure_G=8.0, G_max=14, n_points=900, seed=9)
res2 = dfit.analyze_dfit(d2.time_min, d2.pressure_psi, d2.rate_bpm)
print(res2.summary())

### Takeaways

- Flow regime is identified from the derivative slope, not wishful straight-line fitting.
- A radial flag is only trusted at late time and over a meaningful span; otherwise pore pressure is left unestimated rather than wrong.
- This mirrors the central message of SPE-169539-PA.